# Temporal Analytics

**Dataset:** `samples.bakehouse.sales_transactions`, `samples.bakehouse.sales_customers`

**Difficulty:** Hard

**Topics:** `row_number`, `lag`, `date_trunc`, `datediff`, `months_between`, `Window`, cohort retention, gaps-and-islands, churn segmentation, WoW momentum

> All six problems revolve around **when** things happened — the hardest class of analytical
> questions in practice. Each one requires combining window functions, date arithmetic, and
> multi-step aggregation pipelines.

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

transactions = spark.table("samples.bakehouse.sales_transactions")
customers    = spark.table("samples.bakehouse.sales_customers")

## Learn — Temporal Patterns Reference

| Pattern | Key functions |
|---------|---------------|
| First/last event per entity | `Window.partitionBy(id).orderBy(ts)` + `F.row_number()` or `F.rank()` |
| Time between consecutive events | `F.lag(col).over(w)` then `F.datediff` |
| Truncate to week / month / year | `F.date_trunc('week', col)` — returns a timestamp at period start |
| Period-over-period comparison | `F.lag('metric', 1).over(Window.orderBy('period'))` |
| Cohort month | `F.date_trunc('month', F.min('dateTime'))` per customer |
| Months between two dates | `F.months_between(later, earlier).cast('int')` |
| Gaps-and-islands (streaks) | `date - row_number` trick — equal dates form one island |
| Lapsed / active segmentation | compare `F.max(dateTime)` per customer against a reference date |

**Docs:**
- [Window functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/window.html)
- [F.lag](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.lag.html)
- [F.date_trunc](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.date_trunc.html)
- [F.datediff](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.datediff.html)

> **Gaps-and-islands trick:** assign a `row_number` to each purchase date per customer (ordered by
> date), then compute `date - row_number days`. Consecutive days produce the same value, so
> grouping by that value gives you each streak as an island.

In [ ]:
# Run this example first — then solve the problems below.
# NOTE: this example is NOT a solution to any problem.

# Pattern 1: label each transaction as first or nth purchase per customer
w = Window.partitionBy("customerID").orderBy("dateTime")
transactions.withColumn("purchase_num", F.row_number().over(w)).select(
    "customerID", "dateTime", "totalPrice", "purchase_num"
).show(5)

# Pattern 2: days between consecutive purchases (lag)
transactions.withColumn(
    "prev_date", F.lag(F.to_date("dateTime")).over(w)
).withColumn(
    "gap_days", F.datediff(F.to_date("dateTime"), F.col("prev_date"))
).select("customerID", "dateTime", "prev_date", "gap_days").show(5)

## Problem 1 — First vs Repeat Purchase Revenue Split

Label every transaction as either `first_purchase` (the customer's earliest transaction)
or `repeat` (all subsequent ones). Then compute a **summary** grouped by `purchase_type`:
total revenue, transaction count, and average order value.

Business context: the growth team needs to know what fraction of revenue comes from
first-time buyers vs returning customers — this drives the budget split between
acquisition spend and retention programmes.

**Step outline:**
1. Use `F.row_number()` over `Window.partitionBy('customerID').orderBy('dateTime')`.
2. When `row_number == 1` → `'first_purchase'`, else → `'repeat'`.
3. Group by `purchase_type` and aggregate.

**Expected output columns:**
- `purchase_type` — `'first_purchase'` or `'repeat'`
- `transaction_count`
- `total_revenue`
- `avg_order_value` — rounded to 2 decimal places

**Sorted:** by `transaction_count` descending.

**Docs:** [F.row_number](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.row_number.html)

In [ ]:
# Problem 1 - write your solution here
# Assign your result to: result_1

result_1 = None  # replace this

In [ ]:
# ── Tests for Problem 1 ──────────────────────────────────────────
assert result_1 is not None, "result_1 is None - did you forget to assign your DataFrame?"
assert hasattr(result_1, 'columns'), "result_1 must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
for col in ['purchase_type', 'transaction_count', 'total_revenue', 'avg_order_value']:
    assert col in cols, f"Missing column: {col}"
cnt = result_1.count()
assert cnt == 2, f"Expected exactly 2 rows (first_purchase + repeat), got {cnt}"
types = {r['purchase_type'] for r in result_1.collect()}
assert 'first_purchase' in types, "Missing purchase_type 'first_purchase'"
assert 'repeat' in types, "Missing purchase_type 'repeat'"
rows = result_1.collect()
assert all(r['total_revenue'] > 0 for r in rows), "total_revenue must be positive"
assert all(r['avg_order_value'] > 0 for r in rows), "avg_order_value must be positive"
# Repeat count should exceed first_purchase count (every customer has 1 first and potentially many repeats)
repeat_cnt = next(r['transaction_count'] for r in rows if r['purchase_type'] == 'repeat')
first_cnt  = next(r['transaction_count'] for r in rows if r['purchase_type'] == 'first_purchase')
assert repeat_cnt > first_cnt, "Expected more repeat transactions than first purchases"
print(f"Problem 1 passed ✓  first_purchase={first_cnt} txns, repeat={repeat_cnt} txns")

## Problem 2 — Inter-Purchase Gap Analysis

For each customer compute the **time in days between consecutive purchases** using
`F.lag()`. Then aggregate per customer to get:
- `purchase_count` — total number of purchases
- `avg_gap_days` — mean days between consecutive orders (null for single-purchase customers)
- `max_gap_days` — longest single gap between orders

Finally, filter to customers with at least **2 purchases** and sort by `max_gap_days` descending
to find customers who went the longest without re-ordering.

Business context: customers with large purchase gaps are prime candidates for re-engagement
campaigns. The CRM team uses this to trigger automated win-back emails at the right moment.

**Step outline:**
1. `Window.partitionBy('customerID').orderBy('dateTime')` — per-customer chronological window.
2. `F.lag(F.to_date('dateTime')).over(w)` → `prev_date`.
3. `F.datediff(F.to_date('dateTime'), prev_date)` → `gap_days` (null on the first row).
4. Group by `customerID`, compute avg/max of `gap_days`, count rows.
5. Filter `purchase_count >= 2`, sort `max_gap_days` desc.

**Expected output columns:**
- `customerID`
- `purchase_count`
- `avg_gap_days` — rounded to 1 decimal place
- `max_gap_days`

**Sorted:** by `max_gap_days` descending.

In [ ]:
# Problem 2 - write your solution here
# Assign your result to: result_2

result_2 = None  # replace this

In [ ]:
# ── Tests for Problem 2 ──────────────────────────────────────────
assert result_2 is not None, "result_2 is None - did you forget to assign your DataFrame?"
assert hasattr(result_2, 'columns'), "result_2 must be a Spark DataFrame"
cols = [c.lower() for c in result_2.columns]
for col in ['customerid', 'purchase_count', 'avg_gap_days', 'max_gap_days']:
    assert col in cols, f"Missing column: {col}"
cnt = result_2.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
# All customers should have >= 2 purchases
low = result_2.filter(F.col('purchase_count') < 2).count()
assert low == 0, f"Found {low} customers with purchase_count < 2 — filter them out"
# No null avg_gap_days (they have >= 2 purchases so lag will produce values)
null_avg = result_2.filter(F.col('avg_gap_days').isNull()).count()
assert null_avg == 0, f"Found {null_avg} null avg_gap_days values"
# max_gap >= avg_gap for all rows
bad = result_2.filter(F.col('max_gap_days') < F.col('avg_gap_days')).count()
assert bad == 0, f"Found {bad} rows where max_gap_days < avg_gap_days"
# Sorted descending
gaps = [r['max_gap_days'] for r in result_2.collect()]
assert gaps == sorted(gaps, reverse=True), "Results must be sorted by max_gap_days descending"
print(f"Problem 2 passed ✓  ({cnt} customers, max single gap={gaps[0]} days)")

## Problem 3 — Active vs Lapsed Customer Segmentation

Segment customers into **`active`** (purchased within 90 days of the dataset's most recent
transaction) and **`lapsed`** (last purchase was more than 90 days before that cutoff).

Use the **maximum `dateTime`** in the transactions table as your reference date — do **not**
use `F.current_date()`, because the dataset may not be live.

Return a 2-row summary by segment.

Business context: lapsed customers cost significantly less to reactivate than acquiring new
ones. Knowing the lapsed population size and their historical spend guides the reactivation
budget.

**Step outline:**
1. Compute `reference_date = F.to_date(F.max('dateTime'))` across the whole table (single value).
2. Per customer, find `last_purchase = F.max(F.to_date('dateTime'))` and `total_spend = F.sum('totalPrice')`.
3. Broadcast or cross-join `reference_date`, compute `days_inactive = F.datediff(reference_date, last_purchase)`.
4. Label: `days_inactive <= 90` → `active`, else → `lapsed`.
5. Group by `segment`, aggregate `customer_count` and `avg_total_spend`.

**Expected output columns:**
- `segment` — `'active'` or `'lapsed'`
- `customer_count`
- `avg_total_spend` — rounded to 2 decimal places

**Docs:** [F.datediff](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.datediff.html)

In [ ]:
# Problem 3 - write your solution here
# Assign your result to: result_3

result_3 = None  # replace this

In [ ]:
# ── Tests for Problem 3 ──────────────────────────────────────────
assert result_3 is not None, "result_3 is None - did you forget to assign your DataFrame?"
assert hasattr(result_3, 'columns'), "result_3 must be a Spark DataFrame"
cols = [c.lower() for c in result_3.columns]
for col in ['segment', 'customer_count', 'avg_total_spend']:
    assert col in cols, f"Missing column: {col}"
cnt = result_3.count()
assert cnt == 2, f"Expected exactly 2 rows (active + lapsed), got {cnt}"
segs = {r['segment'] for r in result_3.collect()}
assert 'active' in segs, "Missing segment 'active'"
assert 'lapsed' in segs, "Missing segment 'lapsed'"
rows = result_3.collect()
assert all(r['customer_count'] > 0 for r in rows), "customer_count must be > 0 for each segment"
assert all(r['avg_total_spend'] > 0 for r in rows), "avg_total_spend must be > 0"
total_customers = sum(r['customer_count'] for r in rows)
assert total_customers > 0, "Total customer count must be > 0"
print(f"Problem 3 passed ✓  active={next(r['customer_count'] for r in rows if r['segment']=='active')}, "
      f"lapsed={next(r['customer_count'] for r in rows if r['segment']=='lapsed')}")

## Problem 4 — Week-over-Week Revenue Momentum

Compute **weekly total revenue**, then calculate the **week-over-week percentage change**
and label each week as `'growth'` (WoW > +10%), `'decline'` (WoW < -10%), or `'stable'`.

Business context: spotting revenue momentum shifts early lets the ops team deploy
promotional offers before a decline becomes a trend.

**Step outline:**
1. `F.date_trunc('week', F.to_timestamp('dateTime'))` → `week_start`.
2. Group by `week_start`, sum `totalPrice` → `weekly_revenue`.
3. `Window.orderBy('week_start')` + `F.lag('weekly_revenue', 1)` → `prev_week_revenue`.
4. `wow_pct_change = (weekly_revenue - prev_week_revenue) / prev_week_revenue * 100` (null for first week).
5. Label with `F.when` / `F.otherwise`.
6. Sort by `week_start` ascending.

**Expected output columns:**
- `week_start` — Monday of each week (DateType)
- `weekly_revenue`
- `prev_week_revenue` — null for the first week
- `wow_pct_change` — rounded to 2 decimal places; null for the first week
- `trend` — `'growth'`, `'decline'`, or `'stable'` (null for the first week is acceptable)

**Sorted:** by `week_start` ascending.

In [ ]:
# Problem 4 - write your solution here
# Assign your result to: result_4

result_4 = None  # replace this

In [ ]:
# ── Tests for Problem 4 ──────────────────────────────────────────
assert result_4 is not None, "result_4 is None - did you forget to assign your DataFrame?"
assert hasattr(result_4, 'columns'), "result_4 must be a Spark DataFrame"
cols = [c.lower() for c in result_4.columns]
for col in ['week_start', 'weekly_revenue', 'prev_week_revenue', 'wow_pct_change', 'trend']:
    assert col in cols, f"Missing column: {col}"
cnt = result_4.count()
assert cnt > 1, f"Expected multiple weeks, got {cnt}"
# First week should have null prev_week_revenue
rows = result_4.orderBy('week_start').collect()
assert rows[0]['prev_week_revenue'] is None, "First week must have null prev_week_revenue"
# Only 3 valid trend values
valid_trends = {'growth', 'decline', 'stable', None}
bad_trends = [r['trend'] for r in rows if r['trend'] not in valid_trends]
assert not bad_trends, f"Invalid trend values: {bad_trends}"
# At least some non-null trends
non_null_trends = [r['trend'] for r in rows if r['trend'] is not None]
assert len(non_null_trends) > 0, "Expected at least some non-null trend values"
# weekly_revenue always positive
assert all(r['weekly_revenue'] > 0 for r in rows), "weekly_revenue must be > 0"
print(f"Problem 4 passed ✓  ({cnt} weeks: {sum(1 for r in rows if r['trend']=='growth')} growth, "
      f"{sum(1 for r in rows if r['trend']=='decline')} decline, "
      f"{sum(1 for r in rows if r['trend']=='stable')} stable)")

## Problem 5 — Monthly Cohort Retention Rate

Build a **cohort retention table**. A cohort is defined by the month a customer made their
**first ever purchase**. For each cohort, track what percentage of customers returned
to make at least one purchase in each subsequent month.

Business context: retention curves show whether the product is improving at keeping
customers engaged. A rising month-2 retention rate across cohorts indicates the loyalty
programme is working.

**Step outline:**
1. Find each customer's `cohort_month`: `F.date_trunc('month', F.min('dateTime'))` grouped by `customerID`.
2. Join back to `transactions` to get every (customer, purchase_month) pair.
   `purchase_month = F.date_trunc('month', dateTime)`.
3. `months_since_cohort = F.round(F.months_between('purchase_month', 'cohort_month')).cast('int')`.
4. Group by `(cohort_month, months_since_cohort)`: count distinct `customerID` → `retained_customers`.
5. Get `cohort_size` = retained_customers where `months_since_cohort == 0`, join back.
6. `retention_rate = F.round(retained_customers / cohort_size * 100, 1)`.
7. Sort by `cohort_month`, `months_since_cohort` ascending.

**Expected output columns:**
- `cohort_month` — first day of the cohort's starting month
- `months_since_cohort` — 0, 1, 2, … (0 = the cohort month itself)
- `cohort_size` — number of customers in this cohort
- `retained_customers` — how many made a purchase this month
- `retention_rate` — percentage, 0–100; cohort month must be 100.0

**Docs:** [F.months_between](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.months_between.html)

In [ ]:
# Problem 5 - write your solution here
# Assign your result to: result_5

result_5 = None  # replace this

In [ ]:
# ── Tests for Problem 5 ──────────────────────────────────────────
assert result_5 is not None, "result_5 is None - did you forget to assign your DataFrame?"
assert hasattr(result_5, 'columns'), "result_5 must be a Spark DataFrame"
cols = [c.lower() for c in result_5.columns]
for col in ['cohort_month', 'months_since_cohort', 'cohort_size', 'retained_customers', 'retention_rate']:
    assert col in cols, f"Missing column: {col}"
cnt = result_5.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
# months_since_cohort must start at 0
min_months = result_5.agg(F.min('months_since_cohort')).collect()[0][0]
assert min_months == 0, f"months_since_cohort must start at 0, got {min_months}"
# At cohort month (months_since_cohort==0) retention_rate should be 100
month_zero = result_5.filter(F.col('months_since_cohort') == 0)
bad_rate = month_zero.filter(F.col('retention_rate') != 100.0).count()
assert bad_rate == 0, f"Found {bad_rate} cohort-month rows where retention_rate != 100.0"
# retention_rate between 0 and 100
out_of_range = result_5.filter((F.col('retention_rate') < 0) | (F.col('retention_rate') > 100)).count()
assert out_of_range == 0, f"Found {out_of_range} rows with retention_rate outside 0-100"
# cohort_size must equal retained_customers at month 0
mismatch = month_zero.filter(F.col('cohort_size') != F.col('retained_customers')).count()
assert mismatch == 0, f"At months_since_cohort=0, cohort_size must equal retained_customers"
print(f"Problem 5 passed ✓  ({cnt} cohort-month rows across {month_zero.count()} cohorts)")

## Problem 6 — Consecutive Purchase Streak (Gaps and Islands)

Find each customer's **longest streak of consecutive calendar days** on which they made
at least one purchase. A streak of 1 means they never bought on two consecutive days.

Business context: high-streak customers are super-fans who may respond well to a
"daily buyer" badge in a gamified loyalty app — the product team wants to know how
many customers have streaks ≥ 5 days.

**Step outline (gaps-and-islands trick):**
1. Extract distinct `(customerID, purchase_date)` pairs — one row per customer per day.
2. Assign `rn = F.row_number()` over `Window.partitionBy('customerID').orderBy('purchase_date')`.
3. Compute `island_key = F.date_sub('purchase_date', F.col('rn').cast('int'))`.
   Consecutive dates will all produce the *same* `island_key`.
4. Group by `(customerID, island_key)` → `streak_length = F.count('*')`.
5. Group by `customerID` → `longest_streak_days = F.max('streak_length')`.
6. Sort by `longest_streak_days` descending.

**Expected output columns:**
- `customerID`
- `longest_streak_days` — ≥ 1 for all customers

**Sorted:** by `longest_streak_days` descending.

**Further reading:** [Gaps and Islands pattern](https://www.sqlshack.com/gaps-and-islands-challenge-sql-server/)

In [ ]:
# Problem 6 - write your solution here
# Assign your result to: result_6

result_6 = None  # replace this

In [ ]:
# ── Tests for Problem 6 ──────────────────────────────────────────
assert result_6 is not None, "result_6 is None - did you forget to assign your DataFrame?"
assert hasattr(result_6, 'columns'), "result_6 must be a Spark DataFrame"
cols = [c.lower() for c in result_6.columns]
assert 'customerid' in cols, "Missing column: customerID"
assert 'longest_streak_days' in cols, "Missing column: longest_streak_days"
cnt = result_6.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
# Each customer should appear exactly once
ids = [r['customerID'] for r in result_6.collect()]
assert len(ids) == len(set(ids)), "Each customerID must appear exactly once"
# All streaks >= 1
bad = result_6.filter(F.col('longest_streak_days') < 1).count()
assert bad == 0, f"Found {bad} customers with longest_streak_days < 1"
# Sorted descending
streaks = [r['longest_streak_days'] for r in result_6.collect()]
assert streaks == sorted(streaks, reverse=True), "Must be sorted by longest_streak_days descending"
top_streak = streaks[0]
super_fans = sum(1 for s in streaks if s >= 5)
print(f"Problem 6 passed ✓  ({cnt} customers, longest streak={top_streak} days, ≥5-day streaks={super_fans})")